# DrugSentiment_POS_NER

## Pipeline projektu: co i dlaczego robimy

1. **Importy i środowisko**  
   Ładujemy biblioteki do NLP, ML, wizualizacji i pracy na danych. Dzięki temu wszystkie kolejne kroki działają w jednym spójnym środowisku.

2. **Pobranie danych (UCI Drug Review, ID=461)**  
   Wczytujemy cechy i target, a potem łączymy je do jednego `DataFrame`. Celem jest mieć pełny zbiór danych (teksty + ocena) w jednym miejscu.

3. **Budowa tekstu wejściowego (`full_review`)**  
   Łączymy `benefitsReview + sideEffectsReview + commentsReview`, aby model widział pełniejszy kontekst opinii użytkownika.

4. **Czyszczenie tekstu (`clean_review`)**  
   Normalizujemy tekst (małe litery, usunięcie znaków specjalnych, redukcja spacji), usuwamy puste rekordy i duplikaty. To ogranicza szum i poprawia jakość cech.

5. **Etykietowanie sentymentu (`label`)**  
   Zamieniamy rating na 3 klasy (`negatywna`, `neutralna`, `pozytywna`). To definiuje problem klasyfikacji wieloklasowej.

6. **Analiza językowa SpaCy (POS/NER)**  
   Tworzymy tokeny, tagi POS i encje NER do analizy jakościowej tekstu oraz budujemy `filtered_review` (np. ADJ/NOUN), by sprawdzić wariant cech oparty o części mowy.

7. **Podział danych i TF-IDF**  
   Dzielimy dane na train/test i tworzymy wektory TF-IDF. To podstawowa reprezentacja numeryczna tekstu dla modeli klasyfikacyjnych.

8. **Balans klas (SMOTE / fallback)**  
   Sprawdzamy nierównowagę klas i stosujemy SMOTE (lub fallback oversampling, jeśli `imblearn` jest niedostępny), aby model lepiej uczył się klas mniejszościowych.

9. **Trening i ewaluacja modeli**  
   Trenujemy modele (LogReg i SVM) na kilku wariantach cech:
   - TF-IDF,
   - TF-IDF + resampling,
   - TF-IDF + POS + resampling.
   Porównujemy metryki (`accuracy`, `precision`, `recall`, `f1`), aby wybrać najlepszy wariant.

10. **Interpretacja modelu**  
    Pokazujemy top słowa (najbardziej wpływowe cechy) dla klas sentymentu, żeby lepiej rozumieć decyzje modelu.

11. **Rozszerzenie cech (`effectiveness`)**  
    Dodajemy liczbową cechę `effectiveness` do macierzy TF-IDF i sprawdzamy, czy poprawia wynik.

12. **Rozszerzona diagnostyka jakości**  
    Oprócz confusion matrix liczymy bogatsze statystyki: `balanced_accuracy`, `MCC`, `Cohen's kappa`, metryki per-klasa, najczęstsze pomyłki klas i rozkład predykcji. Dzięki temu lepiej rozumiemy, gdzie model działa dobrze, a gdzie myli klasy.

13. **Wizualizacja i integracja ze Streamlit**  
    Dodajemy stabilny rendering wykresu (`st.pyplot(fig)`), aby notebook i część interaktywna działały poprawnie i przewidywalnie.

14. **StratifiedKFold (główna ewaluacja 3 klas)**  
    Oprócz pojedynczego train/test split liczymy walidację krzyżową `StratifiedKFold` dla 3 klas (oraz pomocniczo dla 2 klas). Dzięki temu metryki są bardziej stabilne i mniej zależne od jednego podziału danych.

15. **Kalibracja progu dla wariantu 2-klasowego**  
    Dla decyzji końcowej w UI (`godny uwagi` vs `raczej nie`) stroimy próg na osobnym zbiorze kalibracyjnym zamiast używać domyślnego `0.5`. To stabilizuje decyzję modelu i lepiej dopasowuje ją do celu biznesowego (ostrożna rekomendacja).



In [1]:
# 1) Importy (bez instalacji w tej komórce)
# Jeśli czegoś brakuje, uruchom jednorazowo w osobnej komórce:
# %pip install ucimlrepo spacy imbalanced-learn wordcloud
# %pip install -U scikit-learn

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import re
import spacy
import streamlit as st
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import csr_matrix, hstack
from wordcloud import WordCloud

In [2]:
# 2) Pobranie danych UCI + podgląd
# ID = 461 -> Drug Review Dataset (Druglib)
data = fetch_ucirepo(id=461)

df = data.data.features
target = data.data.targets
df = pd.concat([df, target], axis=1)

print(df.shape)
print(df.columns)
print(df.columns.tolist())
df.head()

(4143, 8)
Index(['urlDrugName', 'rating', 'effectiveness', 'sideEffects', 'condition',
       'benefitsReview', 'sideEffectsReview', 'commentsReview'],
      dtype='object')
['urlDrugName', 'rating', 'effectiveness', 'sideEffects', 'condition', 'benefitsReview', 'sideEffectsReview', 'commentsReview']


,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


In [3]:
# 3) full_review + czyszczenie + etykiety
required_cols = ["benefitsReview", "sideEffectsReview", "commentsReview"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Brak wymaganych kolumn: {missing}")

df["full_review"] = (
    df["benefitsReview"].fillna("") + " " +
    df["sideEffectsReview"].fillna("") + " " +
    df["commentsReview"].fillna("")
)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["full_review"].apply(clean_text)
df = df[df["clean_review"] != ""].drop_duplicates(subset=["clean_review"]).reset_index(drop=True)

def label_rating(r):
    if r >= 7:
        return 2
    elif r >= 5:
        return 1
    else:
        return 0

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"]).copy()
df["label"] = df["rating"].apply(label_rating)

print("Balans klas (cały zbiór):")
print(df["label"].value_counts(normalize=True))

Balans klas (cały zbiór):
label
2    0.676607
0    0.219262
1    0.104131
Name: proportion, dtype: float64


In [ ]:
# 4) POS/NER + filtered_review + walidacja stringów
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def process_review(text):
    doc = nlp(text)
    tokens = [t.text for t in doc]
    pos_tags = [(t.text, t.pos_) for t in doc]
    entities = [(e.text, e.label_) for e in doc.ents]
    return tokens, pos_tags, entities

def get_adjectives(text):
    doc = nlp(text)
    return [t.text for t in doc if t.pos_ == "ADJ"]

def filter_pos(text):
    doc = nlp(text)
    return " ".join([t.text for t in doc if t.pos_ in ["ADJ", "NOUN"]])

df["adjectives"] = df["clean_review"].apply(get_adjectives)
df["filtered_review"] = df["clean_review"].apply(filter_pos)

df["filtered_review"] = df["filtered_review"].fillna("").astype(str)
df["clean_review"] = df["clean_review"].fillna("").astype(str)

print(df["filtered_review"].dtype)
print(df["clean_review"].dtype)

sample = df["clean_review"].iloc[0]
tokens, pos_tags, entities = process_review(sample)
print("TOKENS:", tokens[:10])
print("POS:", pos_tags[:10])
print("NER:", entities[:10])

In [ ]:
# 5) Podział danych + TF-IDF
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["label"], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape, X_test_tfidf.shape)

In [ ]:
# 6) SMOTE + funkcja balansu klas + model bazowy

def show_class_balance(y, name="set"):
    ys = pd.Series(y)
    print(f"\nBalans klas ({name}) - counts:")
    print(ys.value_counts().sort_index())
    print(f"Balans klas ({name}) - normalize:")
    print(ys.value_counts(normalize=True).sort_index())

show_class_balance(y_train, "train before resampling")

# Próba SMOTE; jeśli imblearn jest uszkodzony, fallback do prostego oversamplingu
resampler_name = "SMOTE"
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42)

    def fit_resample_fn(X, y):
        return smote.fit_resample(X, y)
except Exception as e:
    from scipy.sparse import vstack

    resampler_name = "Random oversampling fallback"
    print(f"SMOTE niedostępny ({e}). Używam fallback oversampling.")

    def fit_resample_fn(X, y):
        ys = pd.Series(y).reset_index(drop=True)
        max_count = ys.value_counts().max()
        rng = np.random.RandomState(42)

        X_parts = []
        y_parts = []
        for cls in sorted(ys.unique()):
            idx = np.where(ys.values == cls)[0]
            if len(idx) < max_count:
                extra_idx = rng.choice(idx, size=max_count - len(idx), replace=True)
                idx_bal = np.concatenate([idx, extra_idx])
            else:
                idx_bal = idx
            X_parts.append(X[idx_bal])
            y_parts.append(np.full(len(idx_bal), cls))

        X_bal = vstack(X_parts)
        y_bal = np.concatenate(y_parts)

        perm = rng.permutation(len(y_bal))
        return X_bal[perm], y_bal[perm]

X_train_tfidf_smote, y_train_smote = fit_resample_fn(X_train_tfidf, y_train)

show_class_balance(y_train_smote, f"train after {resampler_name}")

model = LogisticRegression(max_iter=500, class_weight="balanced")
model.fit(X_train_tfidf_smote, y_train_smote)

y_pred = model.predict(X_test_tfidf)
print(f"Accuracy (TF-IDF + {resampler_name}):", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["negatywna", "neutralna", "pozytywna"]))
print("Unikalne klasy przewidziane:", np.unique(y_pred))

In [ ]:
# 7) Top słowa dla klas
feature_names = np.array(vectorizer.get_feature_names_out())
for i, label_name in enumerate(["neg", "neu", "pos"]):
    top10 = np.argsort(model.coef_[i])[-10:]
    print(f"\nTop słowa dla {label_name}:")
    print(feature_names[top10])

In [ ]:
# 8) TF-IDF + effectiveness (cecha dodatkowa)
df["effectiveness"] = pd.to_numeric(df["effectiveness"], errors="coerce").fillna(0)
X_extra = csr_matrix(df["effectiveness"].values.reshape(-1, 1))

X_train_combined = hstack([X_train_tfidf, X_extra[X_train.index]])
X_test_combined = hstack([X_test_tfidf, X_extra[X_test.index]])

# opcjonalnie to samo resamplowanie również na cechach połączonych
X_train_combined_smote, y_train_combined_smote = fit_resample_fn(X_train_combined, y_train)

model.fit(X_train_combined_smote, y_train_combined_smote)
y_pred_combined = model.predict(X_test_combined)

print("Accuracy (TF-IDF + effectiveness + SMOTE):", accuracy_score(y_test, y_pred_combined))

In [ ]:
# 9) Stabilny wykres (Notebook + Streamlit)
fig, ax = plt.subplots()
ax.plot([1, 2, 3], [1, 2, 3])
ax.set_title("Example")

import sys
if "streamlit.runtime.scriptrunner" in sys.modules:
    st.pyplot(fig)
else:
    plt.show()

In [ ]:
# 10) Porównanie modeli: Logistic Regression + SVM
# - TF-IDF
# - TF-IDF + SMOTE
# - TF-IDF + POS + SMOTE

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.svm import LinearSVC
from scipy.sparse import hstack, vstack

# --- Resampler (SMOTE lub fallback) ---
def make_resample_fn():
    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=42)

        def _resample(X, y):
            return sm.fit_resample(X, y)

        return _resample, "SMOTE"
    except Exception:
        def _resample(X, y):
            ys = pd.Series(y).reset_index(drop=True)
            max_count = ys.value_counts().max()
            rng = np.random.RandomState(42)

            X_parts, y_parts = [], []
            for cls in sorted(ys.unique()):
                idx = np.where(ys.values == cls)[0]
                if len(idx) < max_count:
                    extra_idx = rng.choice(idx, size=max_count - len(idx), replace=True)
                    idx_bal = np.concatenate([idx, extra_idx])
                else:
                    idx_bal = idx
                X_parts.append(X[idx_bal])
                y_parts.append(np.full(len(idx_bal), cls))

            X_bal = vstack(X_parts)
            y_bal = np.concatenate(y_parts)
            perm = rng.permutation(len(y_bal))
            return X_bal[perm], y_bal[perm]

        return _resample, "FallbackOversampling"

resample_fn, resampler_name = make_resample_fn()

# --- Dane bazowe TF-IDF ---
X_train_base = X_train_tfidf
X_test_base = X_test_tfidf

# --- Dane POS (filtered_review) ---
if "filtered_review" not in df.columns:
    print("'filtered_review' nie istnieje — używam fallback do 'clean_review'.")
    df["filtered_review"] = df["clean_review"].fillna("").astype(str)

X_train_filtered = df.loc[X_train.index, "filtered_review"]
X_test_filtered = df.loc[X_test.index, "filtered_review"]

pos_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words="english")
X_train_pos = pos_vectorizer.fit_transform(X_train_filtered)
X_test_pos = pos_vectorizer.transform(X_test_filtered)

X_train_tfidf_pos = hstack([X_train_base, X_train_pos])
X_test_tfidf_pos = hstack([X_test_base, X_test_pos])

# --- Trening i ocena ---
def train_eval(model, Xtr, ytr, Xte, yte, model_name):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)

    acc = accuracy_score(yte, pred)
    p, r, f1, _ = precision_recall_fscore_support(yte, pred, average="weighted", zero_division=0)

    return {
        "model": model_name,
        "accuracy": round(acc, 4),
        "precision_w": round(p, 4),
        "recall_w": round(r, 4),
        "f1_w": round(f1, 4),
    }

results = []

# Logistic Regression
results.append(train_eval(LogisticRegression(max_iter=500, class_weight="balanced"), X_train_base, y_train, X_test_base, y_test, "LogReg: TF-IDF"))
X_train_sm, y_train_sm = resample_fn(X_train_base, y_train)
results.append(train_eval(LogisticRegression(max_iter=500, class_weight="balanced"), X_train_sm, y_train_sm, X_test_base, y_test, f"LogReg: TF-IDF + {resampler_name}"))
X_train_pos_sm, y_train_pos_sm = resample_fn(X_train_tfidf_pos, y_train)
results.append(train_eval(LogisticRegression(max_iter=500, class_weight="balanced"), X_train_pos_sm, y_train_pos_sm, X_test_tfidf_pos, y_test, f"LogReg: TF-IDF + POS + {resampler_name}"))

# SVM
results.append(train_eval(LinearSVC(class_weight="balanced", random_state=42), X_train_base, y_train, X_test_base, y_test, "SVM: TF-IDF"))
results.append(train_eval(LinearSVC(class_weight="balanced", random_state=42), X_train_sm, y_train_sm, X_test_base, y_test, f"SVM: TF-IDF + {resampler_name}"))
results.append(train_eval(LinearSVC(class_weight="balanced", random_state=42), X_train_pos_sm, y_train_pos_sm, X_test_tfidf_pos, y_test, f"SVM: TF-IDF + POS + {resampler_name}"))

results_df = pd.DataFrame(results).sort_values("f1_w", ascending=False)
print(results_df)


In [ ]:
# 11) Rozszerzone statystyki (poza confusion matrix)
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
)

# Weź najlepszy model z tabeli porównawczej
best_row = results_df.iloc[0]
best_model_name = best_row["model"]
print("Najlepszy model wg f1_w:", best_model_name)

# Odtworzenie predykcji dla najlepszego modelu
is_svm = str(best_model_name).startswith("SVM")
use_pos = "POS" in str(best_model_name)
use_resampling_from_name = ("SMOTE" in str(best_model_name)) or ("FallbackOversampling" in str(best_model_name))

# Wymuszamy ocenę dla zbalansowanych klas (resampling),
# żeby MCC i Kappa były liczone na modelu trenowanym po balansowaniu.
force_balanced_eval = True
use_resampling = True if force_balanced_eval else use_resampling_from_name

print(f"use_resampling_from_name={use_resampling_from_name}, force_balanced_eval={force_balanced_eval}")
if use_pos:
    Xtr = X_train_tfidf_pos
    Xte = X_test_tfidf_pos
else:
    Xtr = X_train_tfidf
    Xte = X_test_tfidf

if use_resampling:
    Xtr_fit, ytr_fit = resample_fn(Xtr, y_train)
else:
    Xtr_fit, ytr_fit = Xtr, y_train

if is_svm:
    from sklearn.svm import LinearSVC
    best_clf = LinearSVC(class_weight="balanced", random_state=42)
else:
    best_clf = LogisticRegression(max_iter=500, class_weight="balanced")

best_clf.fit(Xtr_fit, ytr_fit)
y_pred_best = best_clf.predict(Xte)

# 1) Metryki globalne
print("\n=== Metryki globalne ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_best), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_best), 4))
print("MCC:", round(matthews_corrcoef(y_test, y_pred_best), 4))
print("Cohen Kappa:", round(cohen_kappa_score(y_test, y_pred_best), 4))

# 2) Metryki per-klasa
target_names = ["negatywna", "neutralna", "pozytywna"]
report_dict = classification_report(y_test, y_pred_best, target_names=target_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
print("\n=== Metryki per-klasa ===")
display(report_df)

# 3) Najczęstsze pomyłki klas
cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("\n=== Confusion matrix (counts) ===")
display(cm_df)

errors = []
for i_true in range(cm.shape[0]):
    for i_pred in range(cm.shape[1]):
        if i_true != i_pred and cm[i_true, i_pred] > 0:
            errors.append({
                "true": target_names[i_true],
                "pred": target_names[i_pred],
                "count": int(cm[i_true, i_pred])
            })

errors_df = pd.DataFrame(errors).sort_values("count", ascending=False) if errors else pd.DataFrame(columns=["true", "pred", "count"])
print("\n=== Najczęstsze pomyłki klas ===")
display(errors_df.head(10))

# 4) Rozkład klas: rzeczywiste vs przewidziane
dist_df = pd.DataFrame({
    "true": pd.Series(y_test).value_counts().sort_index(),
    "pred": pd.Series(y_pred_best).value_counts().sort_index(),
}).fillna(0)
dist_df.index = target_names
print("\n=== Rozkład klas (true vs pred) ===")
display(dist_df)


In [ ]:
# 12) BASELINE vs IMPROVED (single-cell comparison)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import re

# Ensure required columns exist
if "clean_review" not in df.columns:
    raise ValueError("Brak kolumny 'clean_review'. Uruchom wcześniejsze komórki preprocessingu.")

# Build clean_v2 if missing (stronger filtering)
if "clean_v2" not in df.columns:
    def make_clean_v2(text):
        t = str(text).lower()
        t = re.sub(r"[^a-z\s]", " ", t)
        # remove very short tokens to reduce noise
        t = " ".join([w for w in t.split() if len(w) > 2])
        return t

    df["clean_v2"] = df["clean_review"].fillna("").apply(make_clean_v2)

# Labels
if "label" not in df.columns:
    raise ValueError("Brak kolumny 'label'. Uruchom komórkę z etykietowaniem.")

# Same split for both pipelines
idx = df.index
train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

y_train = df.loc[train_idx, "label"]
y_test = df.loc[test_idx, "label"]

# ------------------------------
# 1) BASELINE
# ------------------------------
X_train_base_text = df.loc[train_idx, "clean_review"].fillna("").astype(str)
X_test_base_text = df.loc[test_idx, "clean_review"].fillna("").astype(str)

vec_base = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 1),
    stop_words="english"
)
X_train_base = vec_base.fit_transform(X_train_base_text)
X_test_base = vec_base.transform(X_test_base_text)

clf_base = LinearSVC()
clf_base.fit(X_train_base, y_train)
y_pred_base = clf_base.predict(X_test_base)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base, average="weighted")

# ------------------------------
# 2) IMPROVED
# ------------------------------
X_train_imp_text = df.loc[train_idx, "clean_v2"].fillna("").astype(str)
X_test_imp_text = df.loc[test_idx, "clean_v2"].fillna("").astype(str)

vec_imp = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words="english"
)
X_train_imp = vec_imp.fit_transform(X_train_imp_text)
X_test_imp = vec_imp.transform(X_test_imp_text)

clf_imp = LinearSVC()
clf_imp.fit(X_train_imp, y_train)
y_pred_imp = clf_imp.predict(X_test_imp)

acc_imp = accuracy_score(y_test, y_pred_imp)
f1_imp = f1_score(y_test, y_pred_imp, average="weighted")

# ------------------------------
# Top features per class helper
# ------------------------------
def top_features_per_class(clf, vectorizer, class_labels, top_n=10):
    feature_names = np.array(vectorizer.get_feature_names_out())
    tops = {}
    for i, cls in enumerate(class_labels):
        # largest positive coefficients for this class
        top_idx = np.argsort(clf.coef_[i])[-top_n:][::-1]
        tops[cls] = feature_names[top_idx].tolist()
    return tops

class_labels = list(clf_base.classes_)
label_name_map = {0: "negatywna", 1: "neutralna", 2: "pozytywna"}

top_base = top_features_per_class(clf_base, vec_base, class_labels, top_n=10)
top_imp = top_features_per_class(clf_imp, vec_imp, class_labels, top_n=10)

# ------------------------------
# PRINT SECTIONS
# ------------------------------
print("\n" + "=" * 70)
print("BASELINE RESULTS")
print("=" * 70)
print(f"Accuracy: {acc_base:.4f}")
print(f"F1 (weighted): {f1_base:.4f}")
print("\nTop 10 features per class:")
for cls in class_labels:
    cls_name = label_name_map.get(cls, str(cls))
    print(f"- {cls_name} ({cls}): {top_base[cls]}")

print("\n" + "=" * 70)
print("IMPROVED RESULTS")
print("=" * 70)
print(f"Accuracy: {acc_imp:.4f}")
print(f"F1 (weighted): {f1_imp:.4f}")
print("\nTop 10 features per class:")
for cls in class_labels:
    cls_name = label_name_map.get(cls, str(cls))
    print(f"- {cls_name} ({cls}): {top_imp[cls]}")

print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(f"accuracy difference (improved - baseline): {acc_imp - acc_base:+.4f}")
print(f"f1 difference (improved - baseline): {f1_imp - f1_base:+.4f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# =============================
# 0️⃣ Safety: clean_v2 fallback
# =============================
if "clean_v2" not in df.columns:
    import re

    def make_clean_v2(text):
        t = str(text).lower()
        t = re.sub(r"[^a-z\s]", " ", t)
        t = " ".join([w for w in t.split() if len(w) > 2])
        return t

    df["clean_v2"] = df["clean_review"].fillna("").apply(make_clean_v2)

# =============================
# 1️⃣ Split (TEN SAM dla obu!)
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["label"], test_size=0.2, random_state=42
)

# =============================
# 2️⃣ BASELINE
# =============================
vec_base = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 1),
    stop_words="english"
)

X_train_base = vec_base.fit_transform(X_train)
X_test_base = vec_base.transform(X_test)

model_base = LinearSVC()
model_base.fit(X_train_base, y_train)

y_pred_base = model_base.predict(X_test_base)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base, average="weighted")

# =============================
# 3️⃣ IMPROVED
# =============================
X_train_v2 = df.loc[X_train.index, "clean_v2"]
X_test_v2 = df.loc[X_test.index, "clean_v2"]

vec_new = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words="english"
)

X_train_new = vec_new.fit_transform(X_train_v2)
X_test_new = vec_new.transform(X_test_v2)

model_new = LinearSVC()
model_new.fit(X_train_new, y_train)

y_pred_new = model_new.predict(X_test_new)

acc_new = accuracy_score(y_test, y_pred_new)
f1_new = f1_score(y_test, y_pred_new, average="weighted")

# =============================
# 4️⃣ TOP FEATURES
# =============================
def get_top_words(model, vectorizer, label_names, top_n=10):
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = model.coef_

    result = {}
    for i, label in enumerate(label_names):
        top = np.argsort(coefs[i])[-top_n:]
        result[label] = feature_names[top]
    return result

labels = ["neg", "neu", "pos"]

top_base = get_top_words(model_base, vec_base, labels)
top_new = get_top_words(model_new, vec_new, labels)

# =============================
# 5️⃣ PRINT RESULTS
# =============================
print("\n========== BASELINE ==========")
print(f"Accuracy: {acc_base:.4f}")
print(f"F1:       {f1_base:.4f}")
for k, v in top_base.items():
    print(f"\nTop {k}: {v}")

print("\n========== IMPROVED ==========")
print(f"Accuracy: {acc_new:.4f}")
print(f"F1:       {f1_new:.4f}")
for k, v in top_new.items():
    print(f"\nTop {k}: {v}")

print("\n========== COMPARISON ==========")
print(f"Accuracy diff: {acc_new - acc_base:.4f}")
print(f"F1 diff:       {f1_new - f1_base:.4f}")

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image

models = ["Baseline", "Improved"]
accuracy = [0.7009, 0.6960]
f1 = [0.6545, 0.6499]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(x - width/2, accuracy, width, label="Accuracy")
ax.bar(x + width/2, f1, width, label="F1")

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1)
ax.set_title("Baseline vs Improved Model Performance")
ax.set_ylabel("Score")
ax.legend()

# Primary inline render
display(fig)

# Fallback render for environments with flaky inline output
tmp_plot_path = "baseline_vs_improved.png"
fig.savefig(tmp_plot_path, dpi=150, bbox_inches="tight")
print(f"Wykres zapisany do: {tmp_plot_path}")
display(Image(filename=tmp_plot_path))

plt.close(fig)

In [ ]:
# [REMOVED] 3-class Streamlit UI.
# Zostawiamy wyłącznie wersję 2-klasową z kolejnej komórki.

In [ ]:
# 12) Streamlit (wersja 2-klasowa dla użytkownika końcowego)
# Założenie biznesowe: neutralne opinie traktujemy jak "raczej nie" (ostrożne podejście)

import sys

is_streamlit = "streamlit.runtime.scriptrunner" in sys.modules

# Mapowanie 3-klas -> 2-klas (TYLKO dla prezentacji w UI)
# 2 (pozytywna) -> 1 (godny uwagi)
# 0 (negatywna), 1 (neutralna) -> 0 (raczej nie)
ui_label_map = {0: 0, 1: 0, 2: 1}
ui_label_name = {0: "raczej nie", 1: "godny uwagi"}

# Kolumny pomocnicze
if "urlDrugName" in df.columns:
    drug_col = "urlDrugName"
elif "drugName" in df.columns:
    drug_col = "drugName"
else:
    drug_col = df.columns[0]

if "sideEffects" in df.columns:
    side_col = "sideEffects"
elif "sideEffectsReview" in df.columns:
    side_col = "sideEffectsReview"
else:
    side_col = None

# Dane robocze do UI (nie zmieniają wcześniejszych metryk 3-klasowych)
df_ui = df.copy()
df_ui["label_ui"] = df_ui["label"].map(ui_label_map)

all_drugs = sorted(df_ui[drug_col].dropna().astype(str).unique())
selected_drug = all_drugs[0]

if is_streamlit:
    st.title("Ocena leku (2 klasy dla użytkownika)")
    st.caption("Dla bezpieczeństwa neutralne opinie traktujemy jako 'raczej nie'.")
    selected_drug = st.selectbox("Wybierz lek:", all_drugs)
else:
    print("Tryb notebook: wybór domyślny pierwszego leku.")
    print("Neutralne opinie są traktowane jako 'raczej nie'.")

# Filtr dla wybranego leku
drug_reviews = df_ui[df_ui[drug_col].astype(str) == selected_drug].copy()
review_count = len(drug_reviews)

# Decyzja końcowa 2-klasowa (większościowa)
if review_count > 0 and drug_reviews["label_ui"].notna().sum() > 0:
    majority_ui = int(drug_reviews["label_ui"].mode().iloc[0])
    final_decision = ui_label_name[majority_ui]
    decision_counts = drug_reviews["label_ui"].map(ui_label_name).value_counts()

    # Prosty wskaźnik pewności: udział klasy zwycięskiej
    decision_conf = decision_counts.max() / decision_counts.sum()
else:
    final_decision = "brak danych"
    decision_counts = pd.Series(dtype=int)
    decision_conf = 0.0

# Dominujące skutki uboczne
if side_col is not None and side_col in drug_reviews.columns and drug_reviews[side_col].notna().sum() > 0:
    top_side_effect = str(drug_reviews[side_col].mode().iloc[0])
else:
    top_side_effect = "brak danych"

# Render
if is_streamlit:
    c1, c2, c3 = st.columns(3)
    c1.metric("Liczba recenzji", review_count)
    c2.metric("Czy lek godny uwagi?", final_decision)
    c3.metric("Pewność decyzji", f"{decision_conf:.0%}")

    st.write("**Dlaczego 2 klasy?**")
    st.info("Aby nie wprowadzać użytkownika w błąd, opinie neutralne łączymy z negatywnymi jako 'raczej nie'.")

    st.subheader("Rozkład decyzji (2 klasy)")
    if len(decision_counts) > 0:
        st.bar_chart(decision_counts)
    else:
        st.info("Brak danych do rozkładu decyzji.")

    st.subheader("Dominujące skutki uboczne")
    st.write(top_side_effect)

    st.subheader("Przykładowe recenzje")
    cols_to_show = [drug_col]
    if "clean_review" in drug_reviews.columns:
        cols_to_show.append("clean_review")
    if side_col is not None and side_col in drug_reviews.columns:
        cols_to_show.append(side_col)
    st.dataframe(drug_reviews[cols_to_show].head(10), use_container_width=True)
else:
    print("\n=== UI 2-klasowe (podgląd notebook) ===")
    print("Wybrany lek:", selected_drug)
    print("Liczba recenzji:", review_count)
    print("Czy lek godny uwagi?:", final_decision)
    print("Pewność decyzji:", f"{decision_conf:.0%}")
    print("Dominujące skutki uboczne:", top_side_effect)
    if len(decision_counts) > 0:
        print("\nRozkład decyzji (2 klasy):")
        print(decision_counts)


In [ ]:
!python3 -m streamlit run "/Users/turfian/Mastering-NLP-from-Foundations-to-LLMs/drug_sentiment_streamlit.py"

In [ ]:
# 13) Metryki dla 2 klas (jak w Streamlit)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
)
import pandas as pd

# Mapowanie 3-klas -> 2-klas
# 1 = "godny uwagi" (dawna pozytywna)
# 0 = "raczej nie" (dawna neutralna + negatywna)
df_bin = df.copy()
df_bin["label_bin"] = df_bin["label"].apply(lambda x: 1 if int(x) == 2 else 0)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    df_bin["clean_review"],
    df_bin["label_bin"],
    test_size=0.2,
    random_state=42,
    stratify=df_bin["label_bin"],
)

vec_bin = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_b_tfidf = vec_bin.fit_transform(X_train_b)
X_test_b_tfidf = vec_bin.transform(X_test_b)

model_bin = LogisticRegression(max_iter=500, class_weight="balanced")
model_bin.fit(X_train_b_tfidf, y_train_b)

y_pred_b = model_bin.predict(X_test_b_tfidf)

print("=== 2-CLASS METRICS (Streamlit logic) ===")
print("Accuracy:", round(accuracy_score(y_test_b, y_pred_b), 4))
print("Precision:", round(precision_score(y_test_b, y_pred_b, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_b, y_pred_b, zero_division=0), 4))
print("F1:", round(f1_score(y_test_b, y_pred_b, zero_division=0), 4))
print("MCC:", round(matthews_corrcoef(y_test_b, y_pred_b), 4))
print("Cohen Kappa:", round(cohen_kappa_score(y_test_b, y_pred_b), 4))

print("\nClassification report (2 klasy):")
print(classification_report(y_test_b, y_pred_b, target_names=["raczej nie", "godny uwagi"], zero_division=0))

cm_bin = confusion_matrix(y_test_b, y_pred_b, labels=[0, 1])
cm_bin_df = pd.DataFrame(cm_bin, index=["raczej nie", "godny uwagi"], columns=["raczej nie", "godny uwagi"])
print("Confusion matrix (2 klasy):")
display(cm_bin_df)

In [ ]:
# 13b) Kalibracja progu dla 2 klas (wariant Streamlit)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
)
import numpy as np
import pandas as pd

# 3-klasy -> 2-klasy
# 1 = "godny uwagi" (pozytywna), 0 = "raczej nie" (neutralna + negatywna)
df_bin = df.copy()
df_bin["label_bin"] = df_bin["label"].apply(lambda x: 1 if int(x) == 2 else 0)

# train / calibration / test
X_train_b, X_tmp_b, y_train_b, y_tmp_b = train_test_split(
    df_bin["clean_review"],
    df_bin["label_bin"],
    test_size=0.3,
    random_state=42,
    stratify=df_bin["label_bin"],
)

X_cal_b, X_test_b, y_cal_b, y_test_b = train_test_split(
    X_tmp_b,
    y_tmp_b,
    test_size=0.5,
    random_state=42,
    stratify=y_tmp_b,
)

vec_bin_cal = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_b_tfidf = vec_bin_cal.fit_transform(X_train_b)
X_cal_b_tfidf = vec_bin_cal.transform(X_cal_b)
X_test_b_tfidf = vec_bin_cal.transform(X_test_b)

model_bin_cal = LogisticRegression(max_iter=500, class_weight="balanced")
model_bin_cal.fit(X_train_b_tfidf, y_train_b)

# Szukanie najlepszego progu na calibration set
cal_prob = model_bin_cal.predict_proba(X_cal_b_tfidf)[:, 1]
threshold_grid = np.linspace(0.2, 0.8, 25)
rows = []

for thr in threshold_grid:
    pred_thr = (cal_prob >= thr).astype(int)

    tp = int(((pred_thr == 1) & (y_cal_b.values == 1)).sum())
    fp = int(((pred_thr == 1) & (y_cal_b.values == 0)).sum())
    fn = int(((pred_thr == 0) & (y_cal_b.values == 1)).sum())
    tn = int(((pred_thr == 0) & (y_cal_b.values == 0)).sum())

    precision_pos = tp / (tp + fp) if (tp + fp) else 0.0
    recall_pos = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    f1_pos = (2 * precision_pos * recall_pos / (precision_pos + recall_pos)) if (precision_pos + recall_pos) else 0.0
    bal_acc = (recall_pos + specificity) / 2.0

    rows.append(
        {
            "threshold": float(thr),
            "f1_positive": float(f1_pos),
            "balanced_accuracy": float(bal_acc),
            "precision_positive": float(precision_pos),
            "recall_positive": float(recall_pos),
        }
    )

calibration_df = pd.DataFrame(rows).sort_values(
    by=["f1_positive", "balanced_accuracy", "precision_positive"],
    ascending=False,
).reset_index(drop=True)

best_threshold = float(calibration_df.loc[0, "threshold"])
print(f"Najlepszy kalibrowany próg: {best_threshold:.2f}")

# Test po kalibracji
test_prob = model_bin_cal.predict_proba(X_test_b_tfidf)[:, 1]
y_pred_b_cal = (test_prob >= best_threshold).astype(int)

print("=== 2-CLASS METRICS (calibrated threshold) ===")
print("Accuracy:", round(accuracy_score(y_test_b, y_pred_b_cal), 4))
print("Precision:", round(precision_score(y_test_b, y_pred_b_cal, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_b, y_pred_b_cal, zero_division=0), 4))
print("F1:", round(f1_score(y_test_b, y_pred_b_cal, zero_division=0), 4))
print("MCC:", round(matthews_corrcoef(y_test_b, y_pred_b_cal), 4))
print("Cohen Kappa:", round(cohen_kappa_score(y_test_b, y_pred_b_cal), 4))

print("\nTop threshold candidates:")
display(calibration_df.head(10))

print("\nClassification report (2 klasy):")
print(classification_report(y_test_b, y_pred_b_cal, target_names=["raczej nie", "godny uwagi"], zero_division=0))

cm_bin_cal = confusion_matrix(y_test_b, y_pred_b_cal, labels=[0, 1])
cm_bin_cal_df = pd.DataFrame(cm_bin_cal, index=["raczej nie", "godny uwagi"], columns=["raczej nie", "godny uwagi"])
print("Confusion matrix (2 klasy, calibrated):")
display(cm_bin_cal_df)


In [ ]:
!python3 -m streamlit run "/Users/turfian/Mastering-NLP-from-Foundations-to-LLMs/drug_sentiment_streamlit.py" --server.address 127.0.0.1 --server.port 8502

2026-03-26 11:10:17.371 Port 8502 is already in use


In [ ]:
# 14) StratifiedKFold: 3 klasy + 2 klasy
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
)

# Bezpieczne wejście (na wypadek braków po restartach kernela)
if "clean_review" not in df.columns or "label" not in df.columns:
    raise ValueError("Brak kolumn 'clean_review' lub 'label'. Uruchom wcześniejsze komórki preprocessingu.")

text_all = df["clean_review"].fillna("").astype(str)
y3_all = df["label"].astype(int)

# 2-klasowe etykiety pod UI: pozytywna -> 1, neutralna+negatywna -> 0
y2_all = y3_all.apply(lambda x: 1 if int(x) == 2 else 0)

skf3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(text_series, y_series, skf, tag="3-class"):
    rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(text_series, y_series), start=1):
        X_train = text_series.iloc[tr_idx]
        X_test = text_series.iloc[te_idx]
        y_train = y_series.iloc[tr_idx]
        y_test = y_series.iloc[te_idx]

        vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
        X_train_tfidf = vec.fit_transform(X_train)
        X_test_tfidf = vec.transform(X_test)

        clf = LogisticRegression(max_iter=500, class_weight="balanced")
        clf.fit(X_train_tfidf, y_train)
        pred = clf.predict(X_test_tfidf)

        rows.append(
            {
                "fold": fold,
                "accuracy": accuracy_score(y_test, pred),
                "f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
                "balanced_accuracy": balanced_accuracy_score(y_test, pred),
                "mcc": matthews_corrcoef(y_test, pred),
                "kappa": cohen_kappa_score(y_test, pred),
            }
        )

    out = pd.DataFrame(rows)
    print(f"
=== {tag} | fold-by-fold ===")
    display(out)

    print(f"
=== {tag} | mean ± std ===")
    summary = out[["accuracy", "f1_weighted", "balanced_accuracy", "mcc", "kappa"]].agg(["mean", "std"])
    display(summary)

    return out, summary

cv3_df, cv3_summary = run_cv(text_all, y3_all, skf3, tag="3-class")
cv2_df, cv2_summary = run_cv(text_all, y2_all, skf2, tag="2-class (Streamlit mapping)")

print("
Wniosek:")
print("- 3-class: główna ewaluacja projektu (transparentność).")
print("- 2-class: ewaluacja wariantu user-facing dla Streamlit.")



In [ ]:
# 15) Final metrics snapshot z walidacji krzyżowej
# Uruchom po komórce: "StratifiedKFold: 3 klasy + 2 klasy"

required_objs = ["cv3_summary", "cv2_summary"]
missing_objs = [name for name in required_objs if name not in globals()]
if missing_objs:
    raise ValueError(f"Brak wyników CV: {missing_objs}. Uruchom najpierw komórkę KFold.")

snapshot = pd.DataFrame(
    {
        "setup": ["3-class (primary)", "2-class (Streamlit mapping)"],
        "accuracy_mean": [
            float(cv3_summary.loc["mean", "accuracy"]),
            float(cv2_summary.loc["mean", "accuracy"]),
        ],
        "f1_weighted_mean": [
            float(cv3_summary.loc["mean", "f1_weighted"]),
            float(cv2_summary.loc["mean", "f1_weighted"]),
        ],
        "balanced_accuracy_mean": [
            float(cv3_summary.loc["mean", "balanced_accuracy"]),
            float(cv2_summary.loc["mean", "balanced_accuracy"]),
        ],
        "mcc_mean": [
            float(cv3_summary.loc["mean", "mcc"]),
            float(cv2_summary.loc["mean", "mcc"]),
        ],
        "kappa_mean": [
            float(cv3_summary.loc["mean", "kappa"]),
            float(cv2_summary.loc["mean", "kappa"]),
        ],
    }
)

for col in [
    "accuracy_mean",
    "f1_weighted_mean",
    "balanced_accuracy_mean",
    "mcc_mean",
    "kappa_mean",
]:
    snapshot[col] = snapshot[col].round(4)

print("=== FINAL CV METRICS SNAPSHOT ===")
display(snapshot)
